<h1>ERA5 Interactive Analysis with Google Earth Engine</h1>

In [1]:
import ee
import geemap
from IPython.display import *
import ipywidgets as widgets
from ipywidgets import *
from ipyleaflet import WidgetControl
from tqdm.auto import tqdm
from geemap import geojson_to_ee
import cmocean
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import pandas as pd
import datetime
import holoviews as hv
import hvplot.pandas
import panel as pn

In [2]:
try:
    ee.Initialize()
except:
    import httplib2
    ee.Initialize(http_transport=httplib2.Http())

In [3]:
# Map of datasets to bands
band_options = {
    'ERA5 Daily':            ['mean_2m_air_temperature', 'total_precipitation', 'mean_sea_level_pressure'],
    'ERA5 Monthly':          ['mean_2m_air_temperature', 'total_precipitation', 'mean_sea_level_pressure'],
    'Sea Surface Temperature': ['sst', 'ice', 'anom'],
}

# Create the first dropdown
x_widget = widgets.Dropdown(
    options=['ERA5 Daily', 'ERA5 Monthly', 'Sea Surface Temperature'],
    value='ERA5 Daily',
    description='Dataset:'
)

# Create the second dropdown - bands for default dataset
band_widget = widgets.Dropdown(
    options=band_options[x_widget.value],
    description='Band:'
)

# Define a function to update bands based on dataset selection
def update_band_options(change):
    # Update options, resetting value to the first available option
    band_widget.options = band_options[change['new']]
    band_widget.value = band_options[change['new']][0]

# Observe changes in dataset dropdown
x_widget.observe(update_band_options, names='value')

# Display both dropdowns
display(x_widget, band_widget)


Dropdown(description='Dataset:', options=('ERA5 Daily', 'ERA5 Monthly', 'Sea Surface Temperature'), value='ERA…

Dropdown(description='Band:', options=('mean_2m_air_temperature', 'total_precipitation', 'mean_sea_level_press…

In [4]:
d_set = x_widget.value
band = band_widget.value

In [5]:
start_date = ""
end_date = ""

style = {'description_width': 'initial'}
date_widget1 = DatePicker(description='Pick a Start Date', disabled=False, style=style)
date_widget2 = DatePicker(description='Pick an End Date', disabled=False, style=style)

def start_date_handler(change):
    global start_date
    start_date = change.new
def end_date_handler(change):
    global end_date
    end_date = change.new

date_widget1.observe(start_date_handler, names='value')
date_widget2.observe(end_date_handler, names='value')

display(HBox([date_widget1, date_widget2], layout = Layout(display='flex', flex_flow='row', justify_content='space-around', width = "55%")))

In [6]:
datasets = {
    'ERA5 Daily': "ECMWF/ERA5/DAILY",
    'ERA5 Monthly': "ECMWF/ERA5/MONTHLY",
    'Sea Surface Temperature': "NOAA/CDR/OISST/V2_1",
}

dataset = datasets[d_set]

print("You are anlayisng {} data between {} and {}".format(d_set, start_date, end_date))

You are anlayisng ERA5 Daily data between 2010-01-01 and 2015-01-01


In [7]:
def extractMeanTempfromROI(collection, starting_date, ending_date, ROI_geom, bands=band):
  #Returns the mean temperature for a given ROI

  image_collection_clr = collection.select(bands)
  image_collection_timespan = image_collection_clr.filterDate(starting_date, ending_date)
  image_collection_mean_img = image_collection_timespan.reduce(ee.Reducer.mean())
 
  stats = image_collection_mean_img.reduceRegion(**{
    "reducer": ee.Reducer.mean(),
    "geometry": ROI_geom, 
    "scale": 10000,#30, #change this to higher value to speed up processing
    "maxPixels": 1e10
  })

  mean_temp_in_roi_degC = int(stats.getInfo()["{}{}".format(bands,"_mean")])#-275.15
  
  return mean_temp_in_roi_degC

def get_season(date):
    #Returns the season and season mid-date for a given date
    month = date.month
    if month in [12, 1, 2]:
        season = 'DJF'
        mid_date = datetime.date(date.year, 1, 1)
    elif month in [3, 4, 5]:
        season = 'MAM'
        mid_date = datetime.date(date.year, 4, 1)
    elif month in [6, 7, 8]:
        season = 'JJA'
        mid_date = datetime.date(date.year, 7, 1)
    elif month in [9, 10, 11]:
        season = 'SON'
        mid_date = datetime.date(date.year, 10, 1)
    return season, mid_date

In [8]:
from dateutil.rrule import *
from datetime import date

def months_between(start_date, end_date):
  #Returns a list of months between two dates
  return list(
    map(
        date.isoformat,
        rrule(MONTHLY, dtstart=start_date, until=end_date)
        )
    )
  
def days_between(start_date, end_date):
  #Returns a list of days between two dates
  return list(
    map(
        date.isoformat,
        rrule(DAILY, dtstart=start_date, until=end_date)
        )
    )

In [9]:
colors = cmocean.cm.balance(np.linspace(0,1,100))
palette_cols = []
for rgba in colors:
  r=rgba[0]
  g=rgba[1]
  b=rgba[2]
  palette_cols.append(matplotlib.colors.to_hex([r,g,b]))

  ERA5_vis_params = {
  'min': 260,
  'max': 308,
  'palette': palette_cols, 
  'opacity': 0.8
}

ERA5_vis_params_Celcius = {
  'min': 240-273.15,
  'max': 308-273.15,
  'palette': palette_cols
}

In [10]:
image_collection = ee.ImageCollection(dataset).select(band).filter(ee.Filter.date(str(start_date), str(end_date)))
image=image_collection.first()

image_datetime = datetime.date.fromtimestamp(image.get('system:time_start').getInfo()/1000)

In [11]:
#Set up map
Map = geemap.Map(width='100%', )
base_layers = Map.layers

Map.addLayer(image, vis_params=ERA5_vis_params, name="ERA5 Mean 2m Temperature")
Map.add_colorbar(vis_params=ERA5_vis_params_Celcius, label='Temperature (°C)', position='bottomright')

#Empty lists for storing drawing geometries
feat_list, ee_feat_list = [], []

# Get the DrawControl
dc = Map.draw_control

# Handle draw events
def handle_draw(self, action, geo_json):

    Map.addLayer(image, vis_params=ERA5_vis_params, opacity=0.2, name="ERA5 Mean 2m Temperature")
    
    geom = geojson_to_ee(geo_json, False)
    feat_list.append(ee.Feature(geom))
    collection = ee.FeatureCollection(feat_list[-1:])
    clip_image = image.clipToCollection(collection)
    ee_feat_list.append(clip_image)
    
    try:
        Map.remove_drawn_features()
    except:
        pass

    Map.addLayer(clip_image, vis_params=ERA5_vis_params, name='Clipped Data')
    Map.addLayer(collection, {}, 'Drawn Features', False)

dc.on_draw(handle_draw)

In [12]:
display(HTML("<h2>%r</h2>" %  "Displaying {} {} Dataset from {}".format(d_set, band, start_date)), display_id=True)
Map

HTML(value="<h2>'Displaying ERA5 Daily mean_2m_air_temperature Dataset from 2010-01-01'</h2>")

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [13]:
#Return the last drawn area from the map as ROI
roi_dataset = ee_feat_list[-1:]
ROI_geom = dc.last_draw['geometry']
print(ee_feat_list, ee_feat_list[-1:])

[<ee.image.Image object at 0x000001FC08773550>] [<ee.image.Image object at 0x000001FC08773550>]


In [28]:
#Empty lists for storing drawing Temperature data
output_dates, output_temps, output_seasons = [], [], []

image_collection = ee.ImageCollection(dataset)
months = months_between(start_date, end_date)
days = days_between(start_date, end_date)
#seasons = seasons_between(start_date, end_date)

time_type = days# months #days NEED TO ADD CHOICE

#Iterate through the months and extract the mean temperature for each ROI

for i in tqdm(range(len(time_type)-1), desc="Calculating"):
    x = extractMeanTempfromROI(image_collection, time_type[i], time_type[i+1], ROI_geom)
    output_dates.append(datetime.datetime.strptime(time_type[i], '%Y-%m-%d'))
    output_temps.append(x)
print("Finished")

"""
for i in tqdm(range(int((len(time_type)-1)/4)), desc="Calculating"):
    j = i*4
    x = extractMeanTempfromROI(image_collection, time_type[j], time_type[j+3], ROI_geom)
    output_dates.append(get_season(datetime.datetime.strptime(time_type[j], '%Y-%m-%d'))[1])
    output_temps.append(x)
    output_seasons.append(get_season(datetime.datetime.strptime(time_type[j], '%Y-%m-%d'))[0])"""

Calculating:   0%|          | 0/1826 [00:00<?, ?it/s]

Finished


'\nfor i in tqdm(range(int((len(time_type)-1)/4)), desc="Calculating"):\n    j = i*4\n    x = extractMeanTempfromROI(image_collection, time_type[j], time_type[j+3], ROI_geom)\n    output_dates.append(get_season(datetime.datetime.strptime(time_type[j], \'%Y-%m-%d\'))[1])\n    output_temps.append(x)\n    output_seasons.append(get_season(datetime.datetime.strptime(time_type[j], \'%Y-%m-%d\'))[0])'

In [33]:
#Convert lists to dataframe
df = pd.DataFrame({"Date": output_dates, "Temperature": [i-273.15 for i in output_temps]})#, "Season": output_seasons})
df1 = df#.loc[df['Season'] == 'DJF']

#Plot data on interactive line graph and heatmap
linegraph = df1.hvplot(x='Date', y='Temperature', kind='line', width=600, height=300)
heatmap = df.hvplot.heatmap(x='Date.year', y='Date.month', C='Temperature', cmap='reds', width=600, height=300)

#Display graphs in Panel
pn.Row(
    linegraph, 
    heatmap
)

Row
    [0] HoloViews(Curve, height=300, sizing_mode='fixed', width=600)
    [1] HoloViews(HeatMap, height=300, sizing_mode='fixed', width=600)